# Fase 5: DBSCAN Clustering

## Objetivo
Aplicar DBSCAN (Density-Based Spatial Clustering) para identificar clusters baseados em densidade.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.cluster import DBSCAN
from sklearn.neighbors import NearestNeighbors
import matplotlib.pyplot as plt

OUTPUT_DIR = '/home/antonio/Documentos/Machine Learning/atividade_3'
IMG_DIR = f'{OUTPUT_DIR}/imgs'

# Carregar dados
X_std = pd.read_pickle(f'{OUTPUT_DIR}/X_standard.pkl').values
print(f'Dados: {X_std.shape}')

In [ ]:
# K-distance graph
k = 5
nn = NearestNeighbors(n_neighbors=k)
nn.fit(X_std)
distances, indices = nn.kneighbors(X_std)
distances = np.sort(distances[:, k-1])

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(distances, 'b-', linewidth=1)
ax.set_xlabel('Pontos (ordenados)', fontsize=12)
ax.set_ylabel(f'Distancia ao {k}-vizinho mais proximo', fontsize=12)
ax.set_title('Grafico K-Distancia (para encontrar eps)', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'{IMG_DIR}/kdistance_plot.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Testar diferentes eps
print('--- DBSCAN com diferentes eps ---')
for eps in [0.5, 1.0, 1.5, 2.0, 2.5]:
    dbscan = DBSCAN(eps=eps, min_samples=5)
    labels = dbscan.fit_predict(X_std)
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    n_noise = list(labels).count(-1)
    print(f'eps={eps}: {n_clusters} clusters, {n_noise} noise points')

In [ ]:
# Aplicar DBSCAN final (usar eps=1.5 como escolha razoavel)
optimal_eps = 1.5
dbscan = DBSCAN(eps=optimal_eps, min_samples=5)
dbscan_labels = dbscan.fit_predict(X_std)

n_clusters = len(set(dbscan_labels)) - (1 if -1 in dbscan_labels else 0)
n_noise = list(dbscan_labels).count(-1)
print(f'\nDBSCAN final (eps={optimal_eps}):')
print(f'  Clusters: {n_clusters}')
print(f'  Noise: {n_noise} ({n_noise/len(dbscan_labels)*100:.1f}%)')

for i in range(n_clusters):
    print(f'  Cluster {i}: {list(dbscan_labels).count(i)} points')

In [ ]:
# Salvar labels
np.save(f'{OUTPUT_DIR}/dbscan_labels.npy', dbscan_labels)
print(f'Labels salvos')